### pip install torchvision nltk


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from collections import Counter

from pycocotools.coco import COCO
from torch.nn.utils.rnn import pad_sequence

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
# print(torch.version.cuda)
coco = COCO('02_Assignment/coco/annotations/captions_train2017.json')
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on: {device}")

Working on: mps


In [6]:
IMAGE_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomCrop((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

 ### Vocabulary class

In [7]:
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold

        self.itos = {0 : "<pad>", 1 : "<sos>", 2 : "<eos>", 3 : "<unk>"}
        self.stoi = {"<pad>" : 0, "<sos>" : 1, "<eos>" : 2, "<unk>" : 3}

    def __len__(self):
        return len(self.itos)
    
  
    # def tokenize_eng(text):
        # return nltk.tokenize.word_tokenize(text.lower())
    
    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in nltk.tokenize.word_tokenize(sentence.lower()):
                frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
    
    def numericalize(self, text):
        tokenized_text = nltk.tokenize.word_tokenize(text.lower())

        return[
            self.stoi["<sos>"]
            if token in self.stoi
            else self.stoi["<unk>"]
            for token in tokenized_text
        ]


    
    

### COCO Dataset class
- connecting images and captions into one pipeling

In [10]:
class COCOCaptionsDataset(Dataset):
    def __init__(self, root, annFile, vocab, transform=None):
        self.root = root
        self.coco = COCO(annFile)

        self.ids = list(self.coco.anns.keys())

        self.vocab = vocab
        self.transform = transform

    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        ann_id = self.ids[idx]
        caption = self.coco.anns[ann_id]['caption']

        image_id = self.coco.anns[ann_id]['image_id']
        image_info = self.coco.loadImgs(image_id)[0]

        img_path = os.path.join(self.root, image_info['file_name'])

        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        numericalized_caption = self.vocab.numericalize(caption)

        # numericalized_caption.append(self.vocab.stoi["<sos>"])
        # numericalized_caption += self.vocab.numericalize(caption)

        # numericalized_caption.append(self.vocab.stoi["<eos>"])

        caption_tensor = torch.tensor(numericalized_caption)
        caption_length = len(caption_tensor)

        return image, caption_tensor, caption_length
    
    def collate_fn(batch):
        images = torch.stack([item[0] for item in batch], dim=0)

        captions = [item[1] for item in batch]

        length = torch.tensor([item[2] for item in batch])

        padded_captions = pad_sequence(captions, batch_first=True, padding_value=0)

        attention_masks = (padded_captions == 0)

        return images, padded_captions, length, attention_masks
